In [2]:
#!/usr/bin/env python3
"""
Baixa várias URLs (uma por linha em urls.txt) autenticando com Earthdata (URS/GESDISC)
Salva tudo na pasta "dados/".
"""

import os
import getpass
import time
import requests
from requests.utils import urlparse

# === Classe sugerida pela documentação Earthdata para manter headers ao redirecionar ===
class SessionWithHeaderRedirection(requests.Session):
    AUTH_HOST = 'urs.earthdata.nasa.gov'

    def __init__(self, username, password):
        super().__init__()
        self.auth = (username, password)

    # Mantém (ou remove) cabeçalhos corretamente quando há redirect entre hosts diferentes
    def rebuild_auth(self, prepared_request, response):
        headers = prepared_request.headers
        url = prepared_request.url

        # se houver header 'Authorization', verificar hosts de redirect
        if 'Authorization' in headers:
            original_parsed = requests.utils.urlparse(response.request.url)
            redirect_parsed = requests.utils.urlparse(url)

            # só remove Authorization se não for um redirect envolvendo o host de auth
            if (original_parsed.hostname != redirect_parsed.hostname) and \
               (redirect_parsed.hostname != self.AUTH_HOST) and \
               (original_parsed.hostname != self.AUTH_HOST):
                del headers['Authorization']
        return

# === Configurações ===
URLS_FILE = "/home/matheus/Documentos/GitHub/LACRIO/Dados_satelites/subset_GPM_3IMERGDF_07_20250809_071416_.txt"
DEST_DIR = "/home/matheus/Documentos/GitHub/LACRIO/Dados_satelites/dados_nasa"
SLEEP_BETWEEN = 1.0   # segundos entre downloads (ajuste se quiser)
CHUNK_SIZE = 1024 * 1024  # 1 MB por iteração

# Pegar credenciais (prefira usar variáveis de ambiente em scripts de produção)
username = os.getenv("EARTHDATA_USERNAME") or input("Earthdata username: ").strip()
password = os.getenv("EARTHDATA_PASSWORD") or getpass.getpass("Earthdata password: ")

# Cria pasta destino
os.makedirs(DEST_DIR, exist_ok=True)

# Ler URLs
if not os.path.exists(URLS_FILE):
    raise SystemExit(f"Arquivo de URLs não encontrado: {URLS_FILE}")

with open(URLS_FILE, "r") as f:
    urls = [ln.strip() for ln in f if ln.strip()]

session = SessionWithHeaderRedirection(username, password)
# Identifique seu script/app. Recomendo usar seu e-mail ou nome do projeto.
session.headers.update({"User-Agent": "meu-email@exemplo.com - script-download-gpm"})

failed = []

for idx, url in enumerate(urls, start=1):
    fname = os.path.join(DEST_DIR, os.path.basename(url))
    print(f"[{idx}/{len(urls)}] -> {os.path.basename(url)}")

    if os.path.exists(fname):
        print("   já existe, pulando.")
        continue

    try:
        # faz a requisição com stream; SessionWithHeaderRedirection cuida dos redirects
        r = session.get(url, stream=True, timeout=120)
        r.raise_for_status()

        with open(fname, "wb") as out:
            for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    out.write(chunk)

        print("   salvo:", fname)

    except requests.exceptions.HTTPError as e:
        status = getattr(e.response, "status_code", None) if isinstance(e, requests.exceptions.HTTPError) else None
        print(f"   ❌ HTTP error {status}: {e}")
        failed.append(url)

    except requests.exceptions.RequestException as e:
        print(f"   ❌ Erro de requisição: {e}")
        failed.append(url)

    # Seja educado com o servidor
    time.sleep(SLEEP_BETWEEN)

# grava as falhas
if failed:
    with open("failed_urls.txt", "w") as ff:
        ff.write("\n".join(failed))
    print(f"\nConcluído com {len(failed)} falhas. Veja failed_urls.txt")
else:
    print("\nConcluído com sucesso!")


[1/9923] -> IMERG_V07_ATBD_final.pdf
   já existe, pulando.
[2/9923] -> README.GPM.pdf
   já existe, pulando.
[3/9923] -> 3B-DAY.MS.MRG.3IMERG.19980101-S000000-E235959.V07B.nc4
   já existe, pulando.
[4/9923] -> 3B-DAY.MS.MRG.3IMERG.19980102-S000000-E235959.V07B.nc4
   já existe, pulando.
[5/9923] -> 3B-DAY.MS.MRG.3IMERG.19980103-S000000-E235959.V07B.nc4
   já existe, pulando.
[6/9923] -> 3B-DAY.MS.MRG.3IMERG.19980104-S000000-E235959.V07B.nc4
   já existe, pulando.
[7/9923] -> 3B-DAY.MS.MRG.3IMERG.19980105-S000000-E235959.V07B.nc4
   já existe, pulando.
[8/9923] -> 3B-DAY.MS.MRG.3IMERG.19980106-S000000-E235959.V07B.nc4
   já existe, pulando.
[9/9923] -> 3B-DAY.MS.MRG.3IMERG.19980107-S000000-E235959.V07B.nc4
   já existe, pulando.
[10/9923] -> 3B-DAY.MS.MRG.3IMERG.19980108-S000000-E235959.V07B.nc4
   já existe, pulando.
[11/9923] -> 3B-DAY.MS.MRG.3IMERG.19980109-S000000-E235959.V07B.nc4
   já existe, pulando.
[12/9923] -> 3B-DAY.MS.MRG.3IMERG.19980110-S000000-E235959.V07B.nc4
   já exis

KeyboardInterrupt: 